[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sapiosciences/sapio-py-tutorials/blob/master/11_user_management.ipynb)

In [2]:
from sapiopylib.rest.UserManagerService import VeloxUserManager
from sapiopylib.rest.DataMgmtService import DataMgmtServer
from sapiopylib.rest.User import SapioUser

user: SapioUser = SapioUser(url="https://linux-vm:8443/webservice/api", verify_ssl_cert=False,
                 guid="3c232543-f407-4828-aae5-b33d4cd31fa7",
                 username="yqiao_api", password="Password1!")
user_man: VeloxUserManager = DataMgmtServer.get_user_manager(user)

# Session Information
In addition to credentials and connection details (such as `url`, `guid`, `username`, `password`, and `api_token`), the `SapioUser` session object stores session metadata that becomes available once authenticated.

In a webhook context, `utc_offset_seconds` indicates the timezone offset of the user's browser session (this offset is typically not required outside webhooks).
The logged-in group ID and group name correspond to one of the user groups assigned to the current user. A user session belongs to only one active group at a time. The combination of the user's permissions and active group permissions determines the session's effective privileges. This is particularly relevant for ELN role assignments on notebook experiments and templates.

The active group ID is also evaluated when determining a step's `is_available()` status, which indicates whether the current user session can modify that step.

In [5]:
from sapiopylib.rest.User import UserSessionAdditionalData

session_info: UserSessionAdditionalData = user.session_additional_data

print("My logged in group ID is: " + str(session_info.current_group_id))
print("My logged in group name is: " + str(session_info.current_group_name))
print("My user web session timezone offset is: " + str(session_info.utc_offset_seconds/3600) + " hours")

My logged in group ID is: 7
My logged in group name is: Seamless ELN
My user web session timezone off set is: -4.0 hours


# User Manager
The user manager provides methods for obtaining detailed user information in Sapio.

We can get a list of all available users in the system.

In [6]:
from typing import List

user_name_list: List[str] = user_man.get_user_name_list()
print(user_name_list)

['admin', 'velox_webserver_api', 'yqiao', 'yqiao_api']


We can also retrieve a user's group membership, first name, last name, and email address from the user manager. If you provide criteria, you can narrow the search to return only the results you need.

In [10]:
from sapiopylib.rest.pojo.UserInfo import UserInfoCriteria

user_info = user_man.get_user_info_list(criteria=UserInfoCriteria(username_whitelist=['yqiao']))[0]
print("Name: " + ' '.join([user_info.first_name, user_info.last_name]))
print("Email: " + user_info.email)
print("Group memberships: " + str(user_info.group_map.values()))

Name: Yechen Qiao
Email: someone@somewhere
Group memberships: dict_values(['Admin', 'Lab Managers', 'Lab Techs', 'Requesters', 'Sapio Admin', 'Seamless ELN', 'Seamless LIMS'])


# Group Management
The group manager lets you see the available groups and the members of each group.

You can also query group metadata, which includes the group name, group ID, the client-side plugin path configured for the group's login view, and the data type name of any associated limited user type.

In [18]:
from sapiopylib.rest.pojo.UserInfo import UserInfo, UserGroupInfo

group_man = DataMgmtServer.get_group_manager(user)

info_list: List[UserGroupInfo] = group_man.get_user_group_info_list()
for info in info_list:
    print(str(info.group_id) + ": " + info.group_name)


print("The users that belong to the admin group are: " + str([x.username for x in group_man.get_user_info_map_for_groups(['Admin']).get('Admin')]))

2: Admin
3: Lab Managers
4: Lab Techs
5: Requesters
6: Sapio Admin
7: Seamless ELN
8: Seamless LIMS
The users belong to admin group are: ['yqiao']
